# Featherweight — Phase 2 / Group C: QLoRA training on Colab (T4)

Run **top to bottom** on a **GPU runtime** (Runtime → Change runtime type → T4 GPU).
This is a *thin driver*: all logic lives in the `featherweight` package; this
notebook only orchestrates the Colab-only steps.

> **Caveat (read first):** the GPU-only code below (the Unsloth `train()` call and
> the generation/eval cells) is **authored but Colab-validated** — it follows
> Unsloth's standard Llama-3.1 recipe but exact `SFTConfig` / generation details
> are version-sensitive. The package logic was unit-tested locally.

In [2]:
!nvidia-smi  # confirm a T4 (Turing, SM 75 -> fp16, not bf16)

Thu Jun 18 21:35:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Repo + install

The clone is **idempotent and absolute** so re-running this cell can't nest the
repo inside itself (the cause of earlier path mismatches).

In [3]:
# Idempotent clone to an ABSOLUTE path, then cd there. Safe to re-run.
![ -d /content/Featherweight/.git ] || git clone https://github.com/Nishaant-Soni/Featherweight.git /content/Featherweight
%cd /content/Featherweight

%pip install -q unsloth mlflow      # Unsloth official install; mlflow for tracking
%pip install -q -e .                # the featherweight package (editable)

/content/Featherweight
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for featherweight (pyproject.toml) ... done


> **Editable install + a running kernel:** `pip install -e .` registers the
> package via a `.pth` that Python only reads at startup, so the *current* kernel
> won't see `featherweight` yet. The cell below adds `src/` to the path for this
> session (alternatively: Restart session and re-run from the clone cell).

In [4]:
import sys

sys.path.insert(0, "/content/Featherweight/src")  # so the running kernel sees the package

import torch, featherweight
from featherweight import config

print("torch CUDA:", torch.cuda.is_available(), "| featherweight", featherweight.__version__)
print("ROOT_DIR:", config.ROOT_DIR)  # MUST be /content/Featherweight (no nesting)

torch CUDA: True | featherweight 0.1.0
ROOT_DIR: /content/Featherweight


In [5]:
# Optional: capture the resolved GPU stack for reproducibility (commit this file).
!pip freeze > requirements-colab-train.lock.txt

## 2. Workdir, secrets, and the MLflow backend

In [6]:
from pathlib import Path

# Prefer Google Drive for persistence; fall back to local /content if the mount
# fails (common under the VS Code Colab extension). If you land on local disk,
# /content is ephemeral - download the adapter at the end (last cell).
try:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/featherweight")
except Exception as e:
    print(f"Drive mount unavailable ({e}); using local /content (ephemeral).")
    DRIVE = Path("/content/featherweight_out")

DRIVE.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR = DRIVE / "adapter"
print("workdir:", DRIVE)

Drive mount unavailable ([dfs_ephemeral] Credentials propagation unsuccessful); using local /content (ephemeral).
workdir: /content/featherweight_out


In [7]:
# HF token is OPTIONAL here: the datasets and the unsloth/...bnb-4bit base are all
# ungated. A read token just avoids download rate limits.
import os

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("No HF_TOKEN set - fine (everything used here is ungated).")

No HF_TOKEN set - fine (everything used here is ungated).


In [8]:
# MLflow 3.x errors on the default file:// store (found in Group B), so use sqlite.
import mlflow

mlflow.set_tracking_uri(f"sqlite:///{DRIVE / 'mlflow.db'}")

## 3. Regenerate the formatted data on Colab

Run from the repo root so the data lands in `/content/Featherweight/data`.

In [9]:
!cd /content/Featherweight && python scripts/prep_data.py

xlam formatted:     60000
irrelevance formatted: 7500
train: 66500   heldout: 1000   total: 67500
irrelevance in mix: 7500 (11.1%)
wrote /content/Featherweight/data/train.jsonl and /content/Featherweight/data/heldout.jsonl


## 4. Smoke run (tiny — pipeline sanity before the full run)

`early_stopping=False` so it doesn't try to evaluate within these few steps.

In [10]:
from featherweight.train import sft

sft.train(max_steps=10, limit=64, early_stopping=False, output_dir=DRIVE / "adapter_smoke")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.1-8b-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.6.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/64 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 64 | Num Epochs = 2 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,0.075653


Unsloth: Restored added_tokens_decoder metadata in /content/featherweight_out/adapter_smoke/checkpoint-10/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/featherweight_out/adapter_smoke/tokenizer_config.json.


PosixPath('/content/featherweight_out/adapter_smoke')

## 5. Full QLoRA run

**`max_steps` is a hard ceiling** so the run fits a Colab session (2 full epochs
would be ~16k steps / days on a T4). **Eval-loss early stopping** (on by default)
evaluates a small mixed held-out subset every `config.train.eval_steps` and stops
early once it plateaus, keeping the best checkpoint. Raise `max_steps` if you have
a longer session; tune the budget in Phase 5.

In [11]:
sft.train(max_steps=500, output_dir=ADAPTER_DIR)

==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.1-8b-Instruct-bnb-4bit as a legacy tokenizer.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/66500 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 66,475 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.035834,0.051180
100,0.040124,0.042425
150,0.017987,0.033812
200,0.036197,0.034793
250,0.041156,0.034992
300,0.035039,0.030254
350,0.041907,0.031303
400,0.023901,0.030107
450,0.023612,0.028420
500,0.024057,0.028204


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

PosixPath('/content/featherweight_out/adapter')

In [ ]:
!ls -lh /content/featherweight_out/adapter

In [ ]:
from huggingface_hub import login, create_repo, upload_folder

login(token="hf_xxx")   # your WRITE token

repo_id = "username/featherweight-adapter"   # change username if different
create_repo(repo_id, repo_type="model", private=True, exist_ok=True)

upload_folder(
    folder_path="/content/featherweight_out/adapter",
    repo_id=repo_id,
    repo_type="model",
    )
print("uploaded ->", repo_id)

## 6. Held-out eval: base vs fine-tuned (the Phase 2 verify)

Uses the local-tested `eval.heldout.score`. The generation helper is Colab-validated
(batched left-padded greedy decode); adjust if your stack needs it.

In [ ]:
import torch
from unsloth import FastLanguageModel

from featherweight import config
from featherweight.eval import heldout
from featherweight.utils import tracking

EVAL_N = 200
rows = heldout.load_heldout()[:EVAL_N]


def generate_and_score(model, tokenizer, rows, max_new_tokens=256, batch_size=16):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    preds = []
    for i in range(0, len(rows), batch_size):
        prompts = [r["prompt"] for r in rows[i : i + batch_size]]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        new_tokens = out[:, inputs["input_ids"].shape[1] :]
        preds += tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return heldout.score(preds, [r["completion"] for r in rows])


base_model, base_tok = FastLanguageModel.from_pretrained(
    config.BASE_MODEL_4BIT,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
base_metrics = generate_and_score(base_model, base_tok, rows)
print("BASE:", base_metrics)

In [ ]:
# Reload the fine-tuned adapter (Unsloth loads the saved LoRA dir) and score it.
ft_model, ft_tok = FastLanguageModel.from_pretrained(
    str(ADAPTER_DIR),
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
ft_metrics = generate_and_score(ft_model, ft_tok, rows)
print("FT:  ", ft_metrics)

with tracking.mlflow_run("heldout-base-vs-ft"):
    tracking.log_metrics({f"base_{k}": v for k, v in base_metrics.items()})
    tracking.log_metrics({f"ft_{k}": v for k, v in ft_metrics.items()})

## 7. Save the adapter

If you trained to local `/content` (Drive mount failed), download the adapter so a
disconnect doesn't lose it. If you used Drive, it is already persisted there.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/featherweight_adapter", "zip", root_dir=ADAPTER_DIR)
print("zipped:", zip_path)
files.download(zip_path)

## Verify (Phase 2 done when all hold)

- [ ] Full run completed (hit `max_steps` or early-stopped) within the T4 VRAM budget.
- [ ] Training + eval loss decreased (MLflow).
- [ ] `ft_metrics["exact_match_accuracy"]` beats `base_metrics["exact_match_accuracy"]`
      by a clear margin; refusal accuracy improved.
- [ ] Adapter saved (Drive) or downloaded (local-disk fallback).